In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

NEXUS_API_KEY = os.getenv("NEXUS_API_KEY")
NEXUS_BASE_URL = os.getenv("NEXUS_BASE_URL")

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [3]:
from langchain_community.document_loaders import WebBaseLoader

C:\Users\user\AppData\Local\Temp\ipykernel_11180\967698044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4.1-nano", api_key = NEXUS_API_KEY, base_url=NEXUS_BASE_URL)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4.1 nano', 'status': 'deprecated', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000017D9496B9E0>, async_client=<openai.resources.chat.completions.completions.As

In [5]:
loader = WebBaseLoader("https://jalammar.github.io/illustrated-transformer/")
loader

In [6]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://jalammar.github.io/illustrated-transformer/', 'title': 'The Illustrated Transformer – Jay Alammar – Visualizing machine learning one concept at a time.', 'description': "Discussions:\nHacker News (65 points, 4 comments), Reddit r/MachineLearning (29 points, 3 comments)\n\n\nTranslations: Arabic, Chinese (Simplified) 1, Chinese (Simplified) 2, French 1, French 2, Italian, Japanese, Korean, Persian, Russian, Spanish 1, Spanish 2, Vietnamese\n\nWatch: MIT’s Deep Learning State of the Art lecture referencing this post\n\nFeatured in courses at Stanford, Harvard, MIT, Princeton, CMU and others\n\n\n \n  \n\n  \n  Update: This post has now become a book! Check out LLM-book.com which contains (Chapter 3) an updated and expanded version of this post speaking about the latest Transformer models and how they've evolved in the seven years since the original Transformer (like Multi-Query Attention and RoPE Positional embeddings).\n  \n \n\n\nIn the previous p

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = splitter.split_documents(docs)

In [8]:
documents

[Document(metadata={'source': 'https://jalammar.github.io/illustrated-transformer/', 'title': 'The Illustrated Transformer – Jay Alammar – Visualizing machine learning one concept at a time.', 'description': "Discussions:\nHacker News (65 points, 4 comments), Reddit r/MachineLearning (29 points, 3 comments)\n\n\nTranslations: Arabic, Chinese (Simplified) 1, Chinese (Simplified) 2, French 1, French 2, Italian, Japanese, Korean, Persian, Russian, Spanish 1, Spanish 2, Vietnamese\n\nWatch: MIT’s Deep Learning State of the Art lecture referencing this post\n\nFeatured in courses at Stanford, Harvard, MIT, Princeton, CMU and others\n\n\n \n  \n\n  \n  Update: This post has now become a book! Check out LLM-book.com which contains (Chapter 3) an updated and expanded version of this post speaking about the latest Transformer models and how they've evolved in the seven years since the original Transformer (like Multi-Query Attention and RoPE Positional embeddings).\n  \n \n\n\nIn the previous p

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key = NEXUS_API_KEY,
    base_url = NEXUS_BASE_URL
)

In [10]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(documents, embeddings)
db

In [11]:
query = "If we do the same self-attention calculation we outlined above"
result = db.similarity_search(query)

result[0].page_content

'If we do the same self-attention calculation we outlined above, just eight different times with different weight matrices, we end up with eight different Z matrices\n\n\n\n\n\nThis leaves us with a bit of a challenge. The feed-forward layer is not expecting eight matrices – it’s expecting a single matrix (a vector for each word). So we need a way to condense these eight down into a single matrix.\nHow do we do that? We concat the matrices then multiply them by an additional weights matrix WO.\n\n\n\n\nThat’s pretty much all there is to multi-headed self-attention. It’s quite a handful of matrices, I realize. Let me try to put them all in one visual so we can look at them in one place\n\n\n\n\n\n\nNow that we have touched upon attention heads, let’s revisit our example from before to see where the different attention heads are focusing as we encode the word “it” in our example sentence:'

In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following questions based only on the provided contest:
    <context>
    {context}
    </context>
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following questions based only on the provided contest:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4.1 nano', 'status': 'deprecated', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs'

In [15]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"If we do the same self-attention calculation we outlined above",
    "context":[Document(page_content = "If we do the same self-attention calculation we outlined above, just eight different times with different weight matrices")]
})


'The provided context suggests performing the same self-attention calculation multiple times—specifically, eight times—each with different weight matrices. This indicates that in the process of self-attention, different parameter sets are used to compute attention scores or transformations, possibly for multiple heads or different attention mechanisms.'

In [17]:
retriever = db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [18]:
response = retrieval_chain.invoke({
    "input":"If we do the same self-attention calculation we outlined above"
})

response["answer"]

"1. **What challenge arises when performing multiple self-attention calculations with different weight matrices?**\n\n   When performing multiple self-attention calculations with different weight matrices, we obtain eight different Z matrices. The challenge is that the feed-forward layer expects a single matrix (or a vector per word), not multiple matrices. Therefore, we need a way to condense these eight matrices into one suitable matrix.\n\n2. **How is this challenge addressed in the process?**\n\n   The solution is to concatenate the eight Z matrices into one and then multiply this combined matrix by an additional weight matrix, \\( W_O \\), to produce a single condensed matrix suitable for the feed-forward layer.\n\n3. **What is the purpose of multiplying each value vector by the softmax score during self-attention?**\n\n   Multiplying each value vector by its corresponding softmax score weights the values such that relevant words' values are kept prominent, while irrelevant words 

In [23]:
response

{'input': 'If we do the same self-attention calculation we outlined above',
 'context': [Document(id='062598e6-3c33-4fd5-a3a0-fe283900f62e', metadata={'source': 'https://jalammar.github.io/illustrated-transformer/', 'title': 'The Illustrated Transformer – Jay Alammar – Visualizing machine learning one concept at a time.', 'description': "Discussions:\nHacker News (65 points, 4 comments), Reddit r/MachineLearning (29 points, 3 comments)\n\n\nTranslations: Arabic, Chinese (Simplified) 1, Chinese (Simplified) 2, French 1, French 2, Italian, Japanese, Korean, Persian, Russian, Spanish 1, Spanish 2, Vietnamese\n\nWatch: MIT’s Deep Learning State of the Art lecture referencing this post\n\nFeatured in courses at Stanford, Harvard, MIT, Princeton, CMU and others\n\n\n \n  \n\n  \n  Update: This post has now become a book! Check out LLM-book.com which contains (Chapter 3) an updated and expanded version of this post speaking about the latest Transformer models and how they've evolved in the se